# 04. Integracion de la base mensual del proyecto

Este notebook integra dos capas que ya fueron validadas en notebooks anteriores:

- la base mensual clima-satelite validada
- la base anual limpia de Agronet a nivel `departamento-anio`

El resultado esperado es una base mensual operativa del proyecto que mantenga la cobertura completa del panel clima-satelite y que incorpore la informacion productiva anual distribuida a nivel mensual de forma metodologicamente consistente.

## Objetivos
- cargar ambas capas intermedias ya validadas
- definir y normalizar un factor mensual de distribucion de produccion
- expandir Agronet anual a nivel `departamento-anio-mes`
- integrar precios historicos del cafe
- unir todo con la base clima-satelite mensual
- generar una salida intermedia lista para la fase final de limpieza y construccion de dataset de modelado

## Resultado esperado
Al finalizar este notebook debemos tener una base mensual integrada que:

1. conserve el panel mensual completo por departamento
2. incorpore las variables anuales de Agronet donde existan datos reales
3. use un `factor_mensual` que sume exactamente `1.0` por año
4. distinga claramente entre dato anual observado y proxy mensual distribuida
5. quede exportada a `BASE_DE_DATOS/INTERMEDIAS`


## Comentario metodologico

Este notebook no resuelve todavia el target final del modelo. Su objetivo es construir la capa mensual integrada que conecta clima, satelite, produccion y precio.

Puntos metodologicos importantes de este paso:

- la produccion mensual calculada aqui es una **proxy operativa**, no una observacion mensual real de Agronet
- la distribucion mensual se normaliza para corregir el problema detectado en el notebook previo del proyecto donde la suma era `0.95`
- la perdida anual observada de Agronet se replica solo como apoyo operativo en meses de cosecha y no debe interpretarse como target mensual observado definitivo


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 160)


def find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'BASE_DE_DATOS').exists():
            return candidate
    raise FileNotFoundError('No se encontro una carpeta BASE_DE_DATOS en la ruta actual ni en sus padres.')


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)
BASE_DATOS = PROJECT_ROOT / 'BASE_DE_DATOS'

INPUT_CLIMA = BASE_DATOS / 'INTERMEDIAS' / 'clima_satelite_mensual_validada.csv'
INPUT_AGRONET = BASE_DATOS / 'INTERMEDIAS' / 'agronet_departamento_anual.csv'
OUTPUT_DIR = BASE_DATOS / 'INTERMEDIAS'
OUTPUT_PATH = OUTPUT_DIR / 'base_mensual_integrada.csv'

print(f'Ruta actual: {CURRENT_DIR}')
print(f'Raiz detectada del proyecto: {PROJECT_ROOT}')
print(f'Archivo clima-satelite: {INPUT_CLIMA}')
print(f'Archivo Agronet anual: {INPUT_AGRONET}')
print(f'Archivo de salida: {OUTPUT_PATH}')


Ruta actual: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2\MODELOS
Raiz detectada del proyecto: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2
Archivo clima-satelite: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2\BASE_DE_DATOS\INTERMEDIAS\clima_satelite_mensual_validada.csv
Archivo Agronet anual: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2\BASE_DE_DATOS\INTERMEDIAS\agronet_departamento_anual.csv
Archivo de salida: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2\BASE_DE_DATOS\INTERMEDIAS\base_mensual_integrada.csv


In [2]:
for required_path in [INPUT_CLIMA, INPUT_AGRONET]:
    if not required_path.exists():
        raise FileNotFoundError(f'No existe el archivo esperado: {required_path}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Rutas validadas correctamente.')


Rutas validadas correctamente.


## Carga de las dos capas de entrada

La base clima-satelite se lee con separador `;` porque fue exportada asi en el notebook anterior. La base anual de Agronet tambien se lee con `;`.


In [3]:
df_clima = pd.read_csv(INPUT_CLIMA, sep=';')
df_agronet = pd.read_csv(INPUT_AGRONET, sep=';')

df_clima['fecha'] = pd.to_datetime(df_clima['fecha'], errors='coerce')

print('Shape clima-satelite:', df_clima.shape)
print('Shape Agronet anual:', df_agronet.shape)
display(df_clima.head(3))
display(df_agronet.head(3))


Shape clima-satelite: (628, 32)
Shape Agronet anual: (36, 19)


,fecha,departamento,precipitation,temp_aire_C,humedad_relativa_pct,soil,def,pet,aet,GDD_cafe,NDVI,EVI,LST_Day_1km,LST_Night_1km,Gpp,Lai_500m,Fpar_500m,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio,fecha_dt,mes,anio,NDVI_anomalia_pct,EVI_anomalia_pct,Gpp_anomalia_pct,Lai_500m_anomalia_pct,precipitation_anomalia_pct,indice_perdida,evento_perdida
0,2000-01-01,Cundinamarca,58.122070,16.040529,80.992858,1030.010331,87.170530,969.042186,881.660989,7.203958,NaN,NaN,NaN,NaN,0.045104,NaN,NaN,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-01-01,1,2000,NaN,NaN,-15.291502,NaN,-6.983227,-15.291502,1
1,2000-02-01,Cundinamarca,102.480240,16.364601,77.921106,1096.139029,4.391722,1015.409971,1010.790421,7.418556,0.275704,0.194953,25.120398,7.237869,0.046012,1.003104,0.286976,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-02-01,2,2000,-50.827307,-44.414822,-2.640640,-43.957202,25.536483,-32.627589,1
2,2000-03-01,Cundinamarca,126.669177,16.723544,77.873874,1071.270787,24.697898,1135.673349,1110.807486,7.621710,0.504174,0.331648,23.907992,11.393505,0.044515,1.484609,0.373975,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-03-01,3,2000,-3.017617,-2.038318,2.444149,-7.367417,-10.835857,-0.870595,0


,anio,departamento,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_t_ha_original,produccion_t,rendimiento_t_ha,correccion_aplicada,motivo_correccion,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,perdida_real_pct,evento_perdida_real
0,2007,Cundinamarca,65,43017.30,48195.69,33729.143730,0.784083,33729.143730,0.784083,0,NaN,0.000000,0.000000,0.629565,0.154518,0.925118,30017.744086,-15.245048,1
1,2008,Cundinamarca,66,43633.35,48989.09,78254.745626,1.793462,35732.355721,0.818923,1,Outlier de rendimiento de Cundinamarca 2008 corregido con promedio 2007-2009,-42522.389905,-0.974539,1.908602,-0.115141,0.925118,30017.744086,-11.479062,0
2,2009,Cundinamarca,69,43475.84,48581.30,37118.057049,0.853763,37118.057049,0.853763,0,NaN,0.000000,0.000000,0.751595,0.102168,0.925118,30017.744086,-7.713077,0


## Verificacion basica de compatibilidad antes del merge

Antes de expandir Agronet, verificamos que las llaves de integracion existan y que los nombres de departamentos sean consistentes entre ambas capas.


In [4]:
compatibilidad = {
    'departamentos_clima': sorted(df_clima['departamento'].dropna().unique().tolist()),
    'departamentos_agronet': sorted(df_agronet['departamento'].dropna().unique().tolist()),
    'anios_clima_min_max': (int(df_clima['anio'].min()), int(df_clima['anio'].max())),
    'anios_agronet_min_max': (int(df_agronet['anio'].min()), int(df_agronet['anio'].max())),
}

print(json.dumps(compatibilidad, indent=2, ensure_ascii=False))


{
  "departamentos_clima": [
    "Cundinamarca",
    "Risaralda"
  ],
  "departamentos_agronet": [
    "Cundinamarca",
    "Risaralda"
  ],
  "anios_clima_min_max": [
    2000,
    2026
  ],
  "anios_agronet_min_max": [
    2007,
    2024
  ]
}


## Distribucion mensual de produccion

Partimos del mismo patrón estacional conceptual usado anteriormente, pero esta vez lo normalizamos para asegurar que la suma anual sea exactamente `1.0`.

Este factor representa una aproximacion operativa de la distribucion de cosecha para la región Centro Sur y se usa solo para repartir la producción anual a nivel mensual.


In [5]:
factor_mensual_raw = {
    1: 0.040,
    2: 0.035,
    3: 0.045,
    4: 0.075,
    5: 0.090,
    6: 0.080,
    7: 0.055,
    8: 0.050,
    9: 0.060,
    10: 0.120,
    11: 0.160,
    12: 0.140,
}

suma_raw = sum(factor_mensual_raw.values())
factor_mensual_norm = {mes: valor / suma_raw for mes, valor in factor_mensual_raw.items()}
meses_cosecha = [4, 5, 6, 10, 11, 12]

factores_df = pd.DataFrame({
    'mes': list(factor_mensual_raw.keys()),
    'factor_mensual_raw': list(factor_mensual_raw.values()),
    'factor_mensual': [factor_mensual_norm[m] for m in factor_mensual_raw.keys()],
})
factores_df['es_mes_cosecha'] = factores_df['mes'].isin(meses_cosecha).astype(int)

display(factores_df)
print('Suma factor original:', suma_raw)
print('Suma factor normalizado:', factores_df['factor_mensual'].sum())


,mes,factor_mensual_raw,factor_mensual,es_mes_cosecha
0,1,0.040,0.042105,0
1,2,0.035,0.036842,0
2,3,0.045,0.047368,0
3,4,0.075,0.078947,1
4,5,0.090,0.094737,1
5,6,0.080,0.084211,1
6,7,0.055,0.057895,0
7,8,0.050,0.052632,0
8,9,0.060,0.063158,0
9,10,0.120,0.126316,1


Suma factor original: 0.9500000000000001
Suma factor normalizado: 0.9999999999999999


## Serie de precios historicos

Se usa la misma serie anual del proyecto para mantener continuidad metodologica con los notebooks previos. La serie llega hasta 2025, por lo que 2026 quedara sin precio a menos que luego se actualice la fuente.


In [6]:
precios_ico = {
    2000: 1900, 2001: 1450, 2002: 1300, 2003: 1500, 2004: 1800,
    2005: 2150, 2006: 2400, 2007: 2750, 2008: 2900, 2009: 2600,
    2010: 3500, 2011: 5900, 2012: 4000, 2013: 3100, 2014: 4200,
    2015: 2900, 2016: 2950, 2017: 2950, 2018: 2700, 2019: 2950,
    2020: 3000, 2021: 4000, 2022: 4500, 2023: 3800, 2024: 6000,
    2025: 8000,
}

FACTOR_PRODUCTOR = 0.78

df_precios = pd.DataFrame([
    {
        'anio': anio,
        'precio_ico_usd_ton': precio,
        'precio_productor_usd_ton': precio * FACTOR_PRODUCTOR,
    }
    for anio, precio in precios_ico.items()
]).sort_values('anio').reset_index(drop=True)

display(df_precios.head(10))
display(df_precios.tail(5))


,anio,precio_ico_usd_ton,precio_productor_usd_ton
0,2000,1900,1482.0
1,2001,1450,1131.0
2,2002,1300,1014.0
3,2003,1500,1170.0
4,2004,1800,1404.0
5,2005,2150,1677.0
6,2006,2400,1872.0
7,2007,2750,2145.0
8,2008,2900,2262.0
9,2009,2600,2028.0


,anio,precio_ico_usd_ton,precio_productor_usd_ton
21,2021,4000,3120.0
22,2022,4500,3510.0
23,2023,3800,2964.0
24,2024,6000,4680.0
25,2025,8000,6240.0


## Expansion mensual de Agronet anual

Creamos una fila por `departamento-anio-mes` para poder integrarla con la grilla mensual clima-satelite.


In [7]:
meses_df = pd.DataFrame({'mes': list(range(1, 13))})
df_agronet_mensual = df_agronet.merge(meses_df, how='cross')

df_agronet_mensual = df_agronet_mensual.merge(
    factores_df,
    on='mes',
    how='left',
)

df_agronet_mensual['produccion_mensual_t'] = df_agronet_mensual['produccion_t'] * df_agronet_mensual['factor_mensual']
df_agronet_mensual['area_mensual_ha'] = df_agronet_mensual['area_cosechada_ha'] * df_agronet_mensual['factor_mensual']

df_agronet_mensual['perdida_real_mensual_pct'] = np.where(
    df_agronet_mensual['es_mes_cosecha'] == 1,
    df_agronet_mensual['perdida_real_pct'],
    np.nan,
)

df_agronet_mensual['fecha'] = pd.to_datetime(
    df_agronet_mensual['anio'].astype(int).astype(str) + '-' + df_agronet_mensual['mes'].astype(int).astype(str).str.zfill(2) + '-01'
)

print('Shape Agronet mensual expandido:', df_agronet_mensual.shape)
display(df_agronet_mensual.head(10))


Shape Agronet mensual expandido: (432, 27)


,anio,departamento,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_t_ha_original,produccion_t,rendimiento_t_ha,correccion_aplicada,motivo_correccion,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,perdida_real_pct,evento_perdida_real,mes,factor_mensual_raw,factor_mensual,es_mes_cosecha,produccion_mensual_t,area_mensual_ha,perdida_real_mensual_pct,fecha
0,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,1,0.040,0.042105,0,1420.174473,1811.254737,NaN,2007-01-01
1,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,2,0.035,0.036842,0,1242.652664,1584.847895,NaN,2007-02-01
2,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,3,0.045,0.047368,0,1597.696282,2037.661579,NaN,2007-03-01
3,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,4,0.075,0.078947,1,2662.827137,3396.102632,-15.245048,2007-04-01
4,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,5,0.090,0.094737,1,3195.392564,4075.323158,-15.245048,2007-05-01
5,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,6,0.080,0.084211,1,2840.348946,3622.509474,-15.245048,2007-06-01
6,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,7,0.055,0.057895,0,1952.739900,2490.475263,NaN,2007-07-01
7,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,8,0.050,0.052632,0,1775.218091,2264.068421,NaN,2007-08-01
8,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,9,0.060,0.063158,0,2130.261709,2716.882105,NaN,2007-09-01
9,2007,Cundinamarca,65,43017.3,48195.69,33729.14373,0.784083,33729.14373,0.784083,0,NaN,0.0,0.0,0.629565,0.154518,0.925118,30017.744086,-15.245048,1,10,0.120,0.126316,1,4260.523419,5433.764211,-15.245048,2007-10-01


## Validacion del factor mensual ya expandido

Aqui verificamos que la nueva distribucion corregida sume exactamente `1.0` por `departamento-anio`.


In [8]:
factor_check = (
    df_agronet_mensual.groupby(['departamento', 'anio'])['factor_mensual']
    .sum()
    .reset_index(name='suma_factor_mensual')
)

display(factor_check.head(10))
print('Min suma factor:', factor_check['suma_factor_mensual'].min())
print('Max suma factor:', factor_check['suma_factor_mensual'].max())


,departamento,anio,suma_factor_mensual
0,Cundinamarca,2007,1.0
1,Cundinamarca,2008,1.0
2,Cundinamarca,2009,1.0
3,Cundinamarca,2010,1.0
4,Cundinamarca,2011,1.0
5,Cundinamarca,2012,1.0
6,Cundinamarca,2013,1.0
7,Cundinamarca,2014,1.0
8,Cundinamarca,2015,1.0
9,Cundinamarca,2016,1.0


Min suma factor: 1.0
Max suma factor: 1.0


## Integracion de precios en la capa mensual Agronet

Los precios son anuales y se agregan por `anio`. Asi quedan disponibles en cada fila mensual de la base integrada.


In [9]:
df_agronet_mensual = df_agronet_mensual.merge(df_precios, on='anio', how='left')
display(df_agronet_mensual[['anio', 'mes', 'departamento', 'precio_ico_usd_ton', 'precio_productor_usd_ton']].head(8))


,anio,mes,departamento,precio_ico_usd_ton,precio_productor_usd_ton
0,2007,1,Cundinamarca,2750,2145.0
1,2007,2,Cundinamarca,2750,2145.0
2,2007,3,Cundinamarca,2750,2145.0
3,2007,4,Cundinamarca,2750,2145.0
4,2007,5,Cundinamarca,2750,2145.0
5,2007,6,Cundinamarca,2750,2145.0
6,2007,7,Cundinamarca,2750,2145.0
7,2007,8,Cundinamarca,2750,2145.0


## Merge final con la base clima-satelite mensual

La base clima-satelite es la capa maestra del panel mensual, por lo que el merge se hace con `left join` desde clima hacia Agronet mensual.


In [10]:
agronet_cols_for_merge = [
    'fecha', 'departamento', 'anio', 'mes',
    'n_municipios',
    'area_cosechada_ha', 'area_sembrada_ha',
    'produccion_t_original', 'rendimiento_t_ha_original',
    'produccion_t', 'rendimiento_t_ha',
    'correccion_aplicada', 'motivo_correccion',
    'delta_produccion_t', 'delta_rendimiento_t_ha',
    'rendimiento_medio_municipal_reportado',
    'dif_rendimiento_calculado_vs_reportado',
    'rendimiento_medio_t_ha', 'produccion_media_t',
    'perdida_real_pct', 'evento_perdida_real',
    'factor_mensual_raw', 'factor_mensual', 'es_mes_cosecha',
    'produccion_mensual_t', 'area_mensual_ha', 'perdida_real_mensual_pct',
    'precio_ico_usd_ton', 'precio_productor_usd_ton',
]

df_integrada = df_clima.merge(
    df_agronet_mensual[agronet_cols_for_merge],
    on=['fecha', 'departamento', 'anio', 'mes'],
    how='left',
)

print('Shape base integrada:', df_integrada.shape)
display(df_integrada.head(5))


Shape base integrada: (628, 57)


,fecha,departamento,precipitation,temp_aire_C,humedad_relativa_pct,soil,def,pet,aet,GDD_cafe,NDVI,EVI,LST_Day_1km,LST_Night_1km,Gpp,Lai_500m,Fpar_500m,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio,fecha_dt,mes,anio,NDVI_anomalia_pct,EVI_anomalia_pct,Gpp_anomalia_pct,Lai_500m_anomalia_pct,precipitation_anomalia_pct,indice_perdida,evento_perdida,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_t_ha_original,produccion_t,rendimiento_t_ha,correccion_aplicada,motivo_correccion,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,perdida_real_pct,evento_perdida_real,factor_mensual_raw,factor_mensual,es_mes_cosecha,produccion_mensual_t,area_mensual_ha,perdida_real_mensual_pct,precio_ico_usd_ton,precio_productor_usd_ton
0,2000-01-01,Cundinamarca,58.122070,16.040529,80.992858,1030.010331,87.170530,969.042186,881.660989,7.203958,NaN,NaN,NaN,NaN,0.045104,NaN,NaN,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-01-01,1,2000,NaN,NaN,-15.291502,NaN,-6.983227,-15.291502,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,Cundinamarca,102.480240,16.364601,77.921106,1096.139029,4.391722,1015.409971,1010.790421,7.418556,0.275704,0.194953,25.120398,7.237869,0.046012,1.003104,0.286976,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-02-01,2,2000,-50.827307,-44.414822,-2.640640,-43.957202,25.536483,-32.627589,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-03-01,Cundinamarca,126.669177,16.723544,77.873874,1071.270787,24.697898,1135.673349,1110.807486,7.621710,0.504174,0.331648,23.907992,11.393505,0.044515,1.484609,0.373975,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-03-01,3,2000,-3.017617,-2.038318,2.444149,-7.367417,-10.835857,-0.870595,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-04-01,Cundinamarca,146.966135,16.678842,81.240824,1098.622452,11.807117,990.875170,978.952966,7.806545,0.438807,0.307035,23.645491,6.576422,0.048026,1.149813,0.300540,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-04-01,4,2000,-15.700155,-12.968768,8.392347,-24.035019,-30.175531,-6.758859,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-05-01,Cundinamarca,238.109657,16.730210,83.152572,1182.556733,0.000000,891.637309,891.514079,7.702863,0.430630,0.286162,23.446429,8.874900,0.049690,1.030764,0.267016,1895.307361,1086.421734,14.454071,10.320921,185.295099,2000-05-01,5,2000,-16.584008,-16.898203,3.782529,-34.932718,13.446331,-9.899894,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Chequeo de cobertura despues del merge

Aqui verificamos que el merge no haya roto la estructura del panel y que los nulos de Agronet aparezcan exactamente donde esperamos: fuera del periodo 2007-2024.


In [11]:
panel_ok = len(df_integrada) == len(df_clima)
nulos_agronet = df_integrada[['produccion_t', 'rendimiento_t_ha', 'perdida_real_pct']].isnull().sum().to_dict()

cobertura_merge = {
    'filas_clima': len(df_clima),
    'filas_integrada': len(df_integrada),
    'panel_preservado': panel_ok,
    'nulos_agronet_clave': nulos_agronet,
}

print(json.dumps(cobertura_merge, indent=2, ensure_ascii=False))


{
  "filas_clima": 628,
  "filas_integrada": 628,
  "panel_preservado": true,
  "nulos_agronet_clave": {
    "produccion_t": 196,
    "rendimiento_t_ha": 196,
    "perdida_real_pct": 196
  }
}


In [12]:
nulos_agronet_por_anio = (
    df_integrada.groupby('anio')[['produccion_t', 'rendimiento_t_ha', 'perdida_real_pct']]
    .apply(lambda x: x.isnull().sum())
)

display(nulos_agronet_por_anio.head(10))
display(nulos_agronet_por_anio.tail(10))


,produccion_t,rendimiento_t_ha,perdida_real_pct
anio,,,
2000,24,24,24
2001,24,24,24
2002,24,24,24
2003,24,24,24
2004,24,24,24
2005,24,24,24
2006,24,24,24
2007,0,0,0
2008,0,0,0


,produccion_t,rendimiento_t_ha,perdida_real_pct
anio,,,
2017,0,0,0
2018,0,0,0
2019,0,0,0
2020,0,0,0
2021,0,0,0
2022,0,0,0
2023,0,0,0
2024,0,0,0
2025,24,24,24


## Reordenamiento y seleccion de columnas finales

Dejamos una estructura clara para la base mensual integrada, agrupando identificacion, clima, satelite, terreno, anomalias y capa Agronet-economica.


In [13]:
final_cols = [
    'fecha', 'departamento', 'mes', 'anio',
    'precipitation', 'temp_aire_C', 'humedad_relativa_pct', 'soil', 'def', 'pet', 'aet', 'GDD_cafe',
    'NDVI', 'EVI', 'LST_Day_1km', 'LST_Night_1km', 'Gpp', 'Lai_500m', 'Fpar_500m',
    'elevacion_media_m', 'elevacion_std_m', 'pendiente_media', 'pendiente_std', 'aspecto_medio',
    'NDVI_anomalia_pct', 'EVI_anomalia_pct', 'Gpp_anomalia_pct', 'Lai_500m_anomalia_pct', 'precipitation_anomalia_pct',
    'indice_perdida', 'evento_perdida',
    'n_municipios',
    'area_cosechada_ha', 'area_sembrada_ha',
    'produccion_t_original', 'rendimiento_t_ha_original',
    'produccion_t', 'rendimiento_t_ha',
    'correccion_aplicada', 'motivo_correccion',
    'delta_produccion_t', 'delta_rendimiento_t_ha',
    'rendimiento_medio_municipal_reportado',
    'dif_rendimiento_calculado_vs_reportado',
    'rendimiento_medio_t_ha', 'produccion_media_t',
    'factor_mensual_raw', 'factor_mensual', 'es_mes_cosecha',
    'produccion_mensual_t', 'area_mensual_ha',
    'perdida_real_pct', 'perdida_real_mensual_pct', 'evento_perdida_real',
    'precio_ico_usd_ton', 'precio_productor_usd_ton',
]

final_cols = [c for c in final_cols if c in df_integrada.columns]
df_integrada_final = df_integrada[final_cols].sort_values(['departamento', 'fecha']).reset_index(drop=True)

print('Shape final de exportacion:', df_integrada_final.shape)
display(df_integrada_final.head(5))


Shape final de exportacion: (628, 56)


,fecha,departamento,mes,anio,precipitation,temp_aire_C,humedad_relativa_pct,soil,def,pet,aet,GDD_cafe,NDVI,EVI,LST_Day_1km,LST_Night_1km,Gpp,Lai_500m,Fpar_500m,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio,NDVI_anomalia_pct,EVI_anomalia_pct,Gpp_anomalia_pct,Lai_500m_anomalia_pct,precipitation_anomalia_pct,indice_perdida,evento_perdida,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_t_ha_original,produccion_t,rendimiento_t_ha,correccion_aplicada,motivo_correccion,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,factor_mensual_raw,factor_mensual,es_mes_cosecha,produccion_mensual_t,area_mensual_ha,perdida_real_pct,perdida_real_mensual_pct,evento_perdida_real,precio_ico_usd_ton,precio_productor_usd_ton
0,2000-01-01,Cundinamarca,1,2000,58.122070,16.040529,80.992858,1030.010331,87.170530,969.042186,881.660989,7.203958,NaN,NaN,NaN,NaN,0.045104,NaN,NaN,1895.307361,1086.421734,14.454071,10.320921,185.295099,NaN,NaN,-15.291502,NaN,-6.983227,-15.291502,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,Cundinamarca,2,2000,102.480240,16.364601,77.921106,1096.139029,4.391722,1015.409971,1010.790421,7.418556,0.275704,0.194953,25.120398,7.237869,0.046012,1.003104,0.286976,1895.307361,1086.421734,14.454071,10.320921,185.295099,-50.827307,-44.414822,-2.640640,-43.957202,25.536483,-32.627589,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-03-01,Cundinamarca,3,2000,126.669177,16.723544,77.873874,1071.270787,24.697898,1135.673349,1110.807486,7.621710,0.504174,0.331648,23.907992,11.393505,0.044515,1.484609,0.373975,1895.307361,1086.421734,14.454071,10.320921,185.295099,-3.017617,-2.038318,2.444149,-7.367417,-10.835857,-0.870595,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-04-01,Cundinamarca,4,2000,146.966135,16.678842,81.240824,1098.622452,11.807117,990.875170,978.952966,7.806545,0.438807,0.307035,23.645491,6.576422,0.048026,1.149813,0.300540,1895.307361,1086.421734,14.454071,10.320921,185.295099,-15.700155,-12.968768,8.392347,-24.035019,-30.175531,-6.758859,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-05-01,Cundinamarca,5,2000,238.109657,16.730210,83.152572,1182.556733,0.000000,891.637309,891.514079,7.702863,0.430630,0.286162,23.446429,8.874900,0.049690,1.030764,0.267016,1895.307361,1086.421734,14.454071,10.320921,185.295099,-16.584008,-16.898203,3.782529,-34.932718,13.446331,-9.899894,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Validaciones finales antes de exportar

Estas validaciones buscan confirmar que la base integrada preserve la grilla mensual, use bien el nuevo factor y mantenga la cobertura esperada de Agronet.


In [14]:
factor_check_final = (
    df_integrada_final[df_integrada_final['anio'].between(2007, 2024)]
    .groupby(['departamento', 'anio'])['factor_mensual']
    .sum()
)

assert df_integrada_final.duplicated(subset=['fecha', 'departamento']).sum() == 0, 'Hay duplicados por fecha-departamento.'
assert len(df_integrada_final) == len(df_clima), 'El merge altero el numero de filas del panel mensual.'
assert set(df_integrada_final['departamento'].unique()) == {'Cundinamarca', 'Risaralda'}, 'Departamentos inesperados en la base integrada.'
assert df_integrada_final['fecha'].min() == pd.Timestamp('2000-01-01'), 'La fecha minima esperada es 2000-01-01.'
assert df_integrada_final['fecha'].max() == pd.Timestamp('2026-02-01'), 'La fecha maxima esperada es 2026-02-01.'
assert np.allclose(factor_check_final.values, 1.0), 'El factor mensual no suma 1.0 en todos los anios con Agronet.'
assert df_integrada_final[df_integrada_final['anio'] < 2007]['produccion_t'].isnull().all(), 'No deberia haber Agronet antes de 2007.'
assert df_integrada_final[df_integrada_final['anio'] > 2024]['produccion_t'].isnull().all(), 'No deberia haber Agronet despues de 2024.'

print('Validaciones finales superadas correctamente.')


Validaciones finales superadas correctamente.


## Exportacion de la base integrada

Se guarda con separador `;` para mantener consistencia con las otras salidas intermedias del proyecto.


In [15]:
df_integrada_final.to_csv(OUTPUT_PATH, index=False, sep=';')
print(f'Archivo exportado en: {OUTPUT_PATH}')


Archivo exportado en: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2\BASE_DE_DATOS\INTERMEDIAS\base_mensual_integrada.csv


## Resumen ejecutivo

Este bloque deja un resumen corto del resultado para enlazar con el siguiente notebook.


In [16]:
resumen = {
    'archivo_salida': str(OUTPUT_PATH),
    'shape_final': df_integrada_final.shape,
    'periodo': f"{df_integrada_final['fecha'].min().date()} a {df_integrada_final['fecha'].max().date()}",
    'factor_mensual_min': float(factor_check_final.min()),
    'factor_mensual_max': float(factor_check_final.max()),
    'panel_preservado': len(df_integrada_final) == len(df_clima),
    'agr_on_2007_2024': int(df_integrada_final[df_integrada_final['anio'].between(2007, 2024)]['produccion_t'].notna().sum()),
    'siguiente_paso': 'usar esta base mensual integrada para la limpieza final y luego derivar la base anual de modelado y la base operativa mensual',
}

print(json.dumps(resumen, indent=2, ensure_ascii=False, default=str))


{
  "archivo_salida": "G:\\.shortcut-targets-by-id\\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\\Proyecto de Analytics\\PROYECTO_V2\\BASE_DE_DATOS\\INTERMEDIAS\\base_mensual_integrada.csv",
  "shape_final": [
    628,
    56
  ],
  "periodo": "2000-01-01 a 2026-02-01",
  "factor_mensual_min": 1.0,
  "factor_mensual_max": 1.0,
  "panel_preservado": true,
  "agr_on_2007_2024": 432,
  "siguiente_paso": "usar esta base mensual integrada para la limpieza final y luego derivar la base anual de modelado y la base operativa mensual"
}


## Siguiente notebook recomendado

El siguiente paso natural es construir `05_construccion_base_anual_modelado.ipynb` o, si prefieres cerrar primero la trazabilidad mensual, un notebook de limpieza y control de calidad final de esta base integrada.
